<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/CalvingPenPipeline/Step01PrepareReferenceClip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "CalvingPenPipeline"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import ProbeVideo, CutClip, GrabFrameAt, ParseTime

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pathlib import Path

GroupDir = Path("/content/drive/Shareddrives/BeastsAndBytes/ANSC 4040 Group Project")
VideoPath = GroupDir / "Calving Pen Videos " / "2026-09-26 00-00-00 Calving Pen 2.avi"
OutputDir = Path("/content/drive/MyDrive/CalvingPenPipeline") / VideoPath.stem

assert VideoPath.exists(), VideoPath
OutputDir.mkdir(parents=True, exist_ok=True)
print(OutputDir)

/content/drive/MyDrive/CalvingPenPipeline/2026-09-26 00-00-00 Calving Pen 2


In [10]:
# 1. Probe: reads only the header, takes seconds.
import json
Info = ProbeVideo(VideoPath)
(OutputDir / "Probe.json").write_text(json.dumps(Info, indent=2))
Info

{'Video': '2026-09-26 00-00-00 Calving Pen 2.avi',
 'Container': 'avi',
 'Codec': 'h264',
 'PixelFormat': 'yuvj420p',
 'Width': 1920,
 'Height': 1080,
 'Fps': 150.0,
 'FrameCount': 2494737,
 'DurationSeconds': 16631.58,
 'Duration': '04:37:12',
 'SizeMb': 3904.1}

In [5]:
# 2. Timeline sheet: one thumbnail every 15 min (~96 for 24 h).
from IPython.display import Image, display
SheetPath = SaveTimelineSheet(VideoPath, OutputDir / "TimelineSheet.jpg", EveryMinutes=15)
display(Image(str(SheetPath)))

KeyboardInterrupt: 

Look at the sheet and note when the cow is in frame and when calving happens. To look closer at a window, rerun the cell below with a different time.

In [ ]:
# Zoom in: one frame at a given time.
import cv2
from IPython.display import Image, display
Frame = GrabFrameAt(VideoPath, "06:30:00", Width=960)
_, Jpg = cv2.imencode(".jpg", Frame)
display(Image(Jpg.tobytes()))

In [ ]:
# 3. Cut the reference clip. Start short (5 min); the later steps get tested on this.
ClipStart = "06:30:00"   # HH:MM:SS into the video; set from the sheet
ClipSeconds = 300

ClipPath = CutClip(VideoPath, ClipStart, ClipSeconds,
                   OutputDir / f"ReferenceClip{ClipStart.replace(':', '')}For{ClipSeconds}s.mp4")
ProbeVideo(ClipPath)